In [1]:
# ================================================================
# ECHO Clean Supervised Baseline
# ================================================================

import os
import sys
import glob
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# Dynamically add project root
PROJECT_ROOT = "/scratch/users/joshua04/ECHO"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.datasets.cardiac_uda import CardiacUDADataset
from src.models.unet import UNet
from src.losses.metrics import (
    dice_score,
    per_class_dice,
    CombinedLoss,
    DeepSupervisionLoss,)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L40S


In [2]:
DATA_ROOT = os.path.join(PROJECT_ROOT, "data", "cardiacUDC_dataset")
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_CLASSES = 5
BASE_CH = 64
IMG_SIZE = 384

BATCH_SIZE = 2
NUM_EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 1e-4

CLASS_WEIGHTS = [0.15, 1.0, 1.0, 1.0, 1.0]

print("Dataset root:", DATA_ROOT)

Dataset root: /scratch/users/joshua04/ECHO/data/cardiacUDC_dataset


In [3]:
print("Discovering dataset...")
for item in sorted(os.listdir(DATA_ROOT)):
    full = os.path.join(DATA_ROOT, item)
    if os.path.isdir(full):
        n = len(glob.glob(os.path.join(full, "*_image.nii.gz")))
        print(f"{item} — {n} volumes")

src_aug = CardiacUDADataset(
    DATA_ROOT,
    domain="G",
    resize=IMG_SIZE,
    augment=True,
    normalize_mode="zscore",)

src_clean = CardiacUDADataset(
    DATA_ROOT,
    domain="G",
    resize=IMG_SIZE,
    augment=False,
    normalize_mode="zscore",)

train_size = int(0.8 * len(src_aug))
val_size = len(src_aug) - train_size

train_ds, _ = random_split(
    src_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),)

_, val_ds = random_split(
    src_clean,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),)

tgt_ds = CardiacUDADataset(
    DATA_ROOT,
    domain="R",
    resize=IMG_SIZE,
    augment=False,
    normalize_mode="zscore",)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
tgt_loader = DataLoader(tgt_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Target:", len(tgt_ds))

Discovering dataset...
Site_G_100 — 97 volumes
Site_G_20 — 21 volumes
Site_G_29 — 29 volumes
Site_R_126 — 84 volumes
Site_R_52 — 52 volumes
Site_R_73 — 70 volumes
label_all_frame — 10 volumes
[CardiacUDA] Domain G: 147 volumes | augment=True | norm=zscore
[CardiacUDA] Domain G: 147 volumes | augment=False | norm=zscore
[CardiacUDA] Domain R: 136 volumes | augment=False | norm=zscore
Train: 117
Val: 30
Target: 136


In [4]:
model = UNet(
    in_ch=1,
    num_classes=NUM_CLASSES,
    base_ch=BASE_CH,
    use_attention=True,
    deep_supervision=True,
    dropout=0.15,).to(device)

n_params = sum(p.numel() for p in model.parameters())
print("Model parameters:", f"{n_params/1e6:.1f}M")

Model parameters: 30.7M


In [5]:
base_loss = CombinedLoss(
    num_classes=NUM_CLASSES,
    ce_weight=1.0,
    dice_weight=1.0,
    boundary_weight=0.5,
    class_weights=CLASS_WEIGHTS,)

loss_fn = DeepSupervisionLoss(base_loss, aux_weights=(0.4, 0.2)).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6,)

print("Loss + Optimizer ready.")

Loss + Optimizer ready.


In [ ]:
best_val = 0.0
best_tgt = 0.0

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    model.train()

    train_loss = 0.0
    train_dice = 0.0
    n_train = 0

    for imgs, masks in tqdm(train_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)

        outputs = model(imgs)
        loss = loss_fn(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        with torch.no_grad():
            main = outputs[0] if isinstance(outputs, tuple) else outputs
            train_dice += dice_score(main, masks, NUM_CLASSES)

        train_loss += loss.item()
        n_train += 1

    scheduler.step()

    train_loss /= n_train
    train_dice /= n_train

    model.eval()
    val_dice = 0.0
    tgt_dice = 0.0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            val_dice += dice_score(model(imgs), masks, NUM_CLASSES)

        for imgs, masks in tgt_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            tgt_dice += dice_score(model(imgs), masks, NUM_CLASSES)

    val_dice /= len(val_loader)
    tgt_dice /= len(tgt_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Loss {train_loss:.4f} | "
        f"TrainDice {train_dice:.4f} | "
        f"Val {val_dice:.4f} | "
        f"Tgt {tgt_dice:.4f}")

    if val_dice > best_val:
        best_val = val_dice
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_val.pth"))

    if tgt_dice > best_tgt:
        best_tgt = tgt_dice
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_target.pth"))

  0%|          | 0/59 [00:00<?, ?it/s]/scratch/users/joshua04/cardiacuda_project/miniconda3/envs/graphecho/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
100%|██████████| 59/59 [00:56<00:00,  1.04it/s]


Epoch 1/30 | Loss 28.2163 | TrainDice 0.1549 | Val 0.2288 | Tgt 0.2467


100%|██████████| 59/59 [00:54<00:00,  1.08it/s]


Epoch 2/30 | Loss 3.1940 | TrainDice 0.3948 | Val 0.5490 | Tgt 0.4416


100%|██████████| 59/59 [01:00<00:00,  1.02s/it]


...

100%|██████████| 59/59 [00:54<00:00,  1.09it/s]


Epoch 7/30 | Loss 1.3891 | TrainDice 0.6734 | Val 0.6551 | Tgt 0.6090


100%|██████████| 59/59 [00:54<00:00,  1.09it/s]


Epoch 8/30 | Loss 1.4403 | TrainDice 0.6552 | Val 0.6969 | Tgt 0.6715


100%|██████████| 59/59 [00:54<00:00,  1.08it/s]
